# 🌍 Merge World Bank Population onto the ICO Coffee Dataset

Merges World Bank population data onto the tidy ICO coffee dataset so **per-capita** consumption can be calculated (raw volume alone is biased toward countries with large populations).

**Input files (upload both in Step 1):**
- `ico_coffee_long.csv` — the tidy ICO coffee data (`Country, Coffee type, Metric, Year, value, is_imputed`)
- `world_population.csv` — World Bank World Development Indicators export, "Population, total" indicator

**Known issue this notebook handles:** country-name mismatches between the two sources (e.g. ICO's `"United States of America"` vs. the World Bank's `"United States"`) — covered with a manual mapping, and anything still unmatched afterward is reported explicitly rather than silently dropped.

**Output:** `ico_coffee_long_with_population.csv` — same rows as the input, plus a `population` column.

## Step 0 — Setup

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
print("pandas:", pd.__version__)

pandas: 2.3.3


## Step 1 — Upload both files

Select `ico_coffee_long.csv` and `world_population.csv` together.

In [3]:
try:
    from google.colab import files
    print("Running in Colab — select ico_coffee_long.csv and world_population.csv...")
    uploaded = files.upload()
    print(f"\nUploaded {len(uploaded)} file(s):", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab — make sure both CSVs are already in the working directory.")

Not running in Colab — make sure both CSVs are already in the working directory.


## Step 2 — Config

The year range and the country-name mapping found during EDA — every mismatch between ICO's naming and the World Bank's naming that showed up when checking for unmatched rows.

In [4]:
ICO_PATH = Path("ico_coffee_long.csv")
POPULATION_PATH = Path("world_population.csv")
OUTPUT_PATH = Path("ico_coffee_long_with_population.csv")

YEAR_RANGE = range(1990, 2020)  # matches the ICO dataset's year coverage

# ICO country name -> World Bank country name, for every mismatch found during EDA
NAME_MAP = {
    "Belgium/Luxembourg": "Belgium",  # approximation -- flagged in README
    "Bolivia (Plurinational State of)": "Bolivia",
    "Congo": "Congo, Rep.",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Democratic Republic of Congo": "Congo, Dem. Rep.",
    "Lao People's Democratic Republic": "Lao PDR",
    "Slovakia": "Slovak Republic",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "United States of America": "United States",
    "Venezuela": "Venezuela, RB",
    "Yemen": "Yemen, Rep.",
}

for p in [ICO_PATH, POPULATION_PATH]:
    print(f"  {'✅' if p.exists() else '❌'} {p}")

  ✅ ico_coffee_long.csv
  ✅ world_population.csv


## Step 3 — Load the population file

The World Bank's CSV export has 4 metadata rows before the real header (`Data Source`, blank, `Last Updated Date`, blank) — `skiprows=4` handles that. Then reshape from wide (one column per year) to long format so it can be merged cleanly.

In [5]:
def load_population(path: Path) -> pd.DataFrame:
    """Load the World Bank population export and reshape it to long format."""
    pop = pd.read_csv(path, skiprows=4)

    year_cols = [str(y) for y in YEAR_RANGE if str(y) in pop.columns]
    pop_long = pop.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=year_cols,
        var_name="Year",
        value_name="population",
    )
    pop_long["Year"] = pop_long["Year"].astype(int)
    pop_long = pop_long.dropna(subset=["population"])
    return pop_long

pop_long = load_population(POPULATION_PATH)
print(f"Loaded population data for {pop_long['Country Name'].nunique()} entities, "
      f"{pop_long['Year'].min()}-{pop_long['Year'].max()}")
pop_long.head()

Loaded population data for 264 entities, 1990-2019


,Country Name,Country Code,Year,population
0,Aruba,ABW,1990,62753.0
1,Africa Eastern and Southern,AFE,1990,311748681.0
2,Afghanistan,AFG,1990,12045660.0
3,Africa Western and Central,AFW,1990,209281291.0
4,Angola,AGO,1990,11626360.0


## Step 4 — Load the ICO data and check name overlap

Before merging, it's worth seeing exactly which ICO country names don't have an exact match in the population data — that's what the `NAME_MAP` in Step 2 was built from.

In [6]:
ico = pd.read_csv(ICO_PATH)
print(f"Loaded {len(ico):,} rows from {ICO_PATH} ({ico['Country'].nunique()} countries)")

ico_countries = set(ico["Country"].unique())
pop_countries = set(pop_long["Country Name"].unique())
unmapped_still = sorted((ico_countries - pop_countries) - set(NAME_MAP.keys()))

print(f"\nICO country names with no exact match AND no entry in NAME_MAP yet: {len(unmapped_still)}")
for name in unmapped_still:
    print(f"  - {name}")

Loaded 8,640 rows from ico_coffee_long.csv (91 countries)

ICO country names with no exact match AND no entry in NAME_MAP yet: 1
  - Unspecified EU stocks


## Step 5 — Merge

Apply the name mapping, then left-merge on `Country + Year` so every ICO row is kept even where no population match exists.

In [9]:
def merge_population(ico: pd.DataFrame, pop_long: pd.DataFrame) -> pd.DataFrame:
    """Merge population onto the ICO data, applying the name mapping first."""
    ico = ico.copy()
    ico["pop_lookup_name"] = ico["Country"].map(lambda c: NAME_MAP.get(c, c))

    merged = ico.merge(
        pop_long[["Country Name", "Year", "population"]],
        left_on=["pop_lookup_name", "Year"],
        right_on=["Country Name", "Year"],
        how="left",
    )
    merged = merged.drop(columns=["Country Name", "pop_lookup_name"])
    return merged

merged = merge_population(ico, pop_long)
print(f"Merged: {len(merged):,} rows (should match the {len(ico):,} input rows exactly)")
merged.head()

Merged: 8,640 rows (should match the 8,640 input rows exactly)


,Country,Coffee type,Metric,Year,value,is_imputed,population
0,Angola,Robusta/Arabica,domestic_consumption_kg,1990.0,1200000.0,False,11626360.0
1,Angola,Robusta/Arabica,domestic_consumption_kg,1991.0,1800000.0,False,12023529.0
2,Angola,Robusta/Arabica,domestic_consumption_kg,1992.0,2100000.0,False,12423712.0
3,Angola,Robusta/Arabica,domestic_consumption_kg,1993.0,1200000.0,False,12827135.0
4,Angola,Robusta/Arabica,domestic_consumption_kg,1994.0,1500000.0,False,13249764.0


## Step 6 — Report anything still unmatched

Even after the mapping, report — never silently drop — anything that didn't find a population match.

In [10]:
still_missing = sorted(merged.loc[merged["population"].isna(), "Country"].unique())
if still_missing:
    print(f"⚠️  {len(still_missing)} entity(ies) unmatched after the name mapping "
          f"(population left as NaN for these):")
    for name in still_missing:
        print(f"   - {name}")
else:
    print("✅ All countries matched.")

⚠️  1 entity(ies) unmatched after the name mapping (population left as NaN for these):
   - Unspecified EU stocks


## Step 7 — Sanity check: Thailand per-capita consumption

A quick spot-check against known figures — Thailand's per-capita coffee consumption should be roughly 0.21 kg in 1990, climbing to roughly 1.17 kg by 2019.

In [11]:
th = merged[(merged.Country == "Thailand") & (merged.Metric == "domestic_consumption_kg")].set_index("Year")
th_percap_1990 = th.loc[1990, "value"] / th.loc[1990, "population"]
th_percap_2019 = th.loc[2019, "value"] / th.loc[2019, "population"]

print(f"Thailand per-capita consumption 1990: {th_percap_1990:.3f} kg")
print(f"Thailand per-capita consumption 2019: {th_percap_2019:.3f} kg")
print(f"Growth: {th_percap_2019 / th_percap_1990:.1f}x")

Thailand per-capita consumption 1990: 0.206 kg
Thailand per-capita consumption 2019: 1.174 kg
Growth: 5.7x


## Step 8 — Save (and download in Colab)

In [12]:
merged.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}  ({len(merged):,} rows)")

try:
    from google.colab import files as _colab_files
    _colab_files.download(str(OUTPUT_PATH))
except ImportError:
    print(f"Not in Colab — find it locally at: {OUTPUT_PATH.resolve()}")

Saved: ico_coffee_long_with_population.csv  (8,640 rows)
Not in Colab — find it locally at: C:\Users\KrissW Laptop\Downloads\merge population data\ico_coffee_long_with_population.csv


## Summary

| Step | Result |
|---|---|
| Population data loaded | 264 entities, 1990-2019 (World Bank WDI) |
| ICO rows | 8,640 (matches input exactly — a left merge, so no rows are lost) |
| Name mismatches fixed | 11 (via `NAME_MAP`) |
| Still unmatched after mapping | 1 — `"Unspecified EU stocks"`, an aggregate row, not a real country |
| Sanity check | Thailand per-capita consumption: 0.206 kg (1990) → 1.174 kg (2019), 5.7x growth — matches prior analysis exactly |
